# Inference Engine — Expert

> **Section 01 — Topic 07 — expert**

**Prerequisites:** `07-inference-engine/advanced.ipynb`

**What you'll learn:**
- Quantization: GPTQ, AWQ, GGUF for efficient inference
- Kernel optimization and inference benchmarking

**What you'll build:**
- A quantization comparison benchmark

## Setup

This notebook requires several quantization libraries. Not all of them may be available on your system — we provide fallbacks and explanations where GPU-specific tools aren't accessible. The core concepts are hardware-independent even if some benchmarks require a CUDA GPU.

In [ ]:
!pip install -q torch transformers matplotlib numpy datasets
# Optional — install if you have a CUDA GPU:
# !pip install -q auto-gptq autoawq llama-cpp-python optimum

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import json
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# We'll use GPT-2 for demonstrations that work on CPU
# and reference larger models for realistic quantization scenarios
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model_fp32 = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model_fp32.eval()

print(f"\nModel: {model_name}")
print(f"Parameters: {sum(p.numel() for p in model_fp32.parameters()):,}")
print(f"Model size (fp32): {sum(p.numel() * p.element_size() for p in model_fp32.parameters()) / 1e6:.1f} MB")

---
## 1. Quantization Deep Dive — GPTQ

Quantization reduces model size and speeds up inference by representing weights with fewer bits. A 16-bit float (fp16) uses 2 bytes per parameter; a 4-bit integer (int4) uses only 0.5 bytes — a 4x reduction. For a 70B parameter model, this means going from 140 GB to 35 GB, making it possible to run on a single GPU.

**GPTQ** (Frantar et al., 2022) is a post-training quantization method that uses a small calibration dataset to determine optimal quantization parameters. The key idea comes from **Optimal Brain Quantization (OBQ)**: quantize weights one at a time, and after each quantization, adjust the remaining weights to compensate for the error introduced.

The algorithm processes each column of the weight matrix:
1. Compute the Hessian (second-order information) using calibration data
2. For each weight in the column, find the nearest quantized value
3. Distribute the quantization error to the remaining un-quantized weights using the Hessian

GPTQ's innovation over OBQ is grouping weights into blocks and processing them in a specific order that makes the computation tractable for billion-parameter models. It can quantize a 175B model in about 4 hours on a single GPU.

Let's first understand quantization fundamentals by implementing symmetric and asymmetric quantization from scratch.

In [ ]:
def symmetric_quantize(tensor, n_bits=8):
    """
    Symmetric quantization: map float range [-max, max] to [-2^(n-1), 2^(n-1)-1].
    The zero point is always 0, so the scale is symmetric around zero.
    """
    qmin = -(2 ** (n_bits - 1))
    qmax = 2 ** (n_bits - 1) - 1
    
    # Scale factor: maps the float range to the integer range
    abs_max = tensor.abs().max()
    scale = abs_max / qmax
    
    # Quantize: round to nearest integer
    quantized = torch.clamp(torch.round(tensor / scale), qmin, qmax).to(torch.int8)
    
    return quantized, scale


def symmetric_dequantize(quantized, scale):
    """Dequantize: multiply by scale to recover approximate float values."""
    return quantized.float() * scale


def asymmetric_quantize(tensor, n_bits=8):
    """
    Asymmetric quantization: map float range [min, max] to [0, 2^n - 1].
    Uses a zero point to handle asymmetric distributions.
    """
    qmin = 0
    qmax = 2 ** n_bits - 1
    
    val_min = tensor.min()
    val_max = tensor.max()
    
    scale = (val_max - val_min) / (qmax - qmin)
    zero_point = torch.round(-val_min / scale).clamp(qmin, qmax).to(torch.int32)
    
    quantized = torch.clamp(
        torch.round(tensor / scale) + zero_point, qmin, qmax
    ).to(torch.uint8)
    
    return quantized, scale, zero_point


def asymmetric_dequantize(quantized, scale, zero_point):
    """Dequantize with zero point."""
    return (quantized.float() - zero_point.float()) * scale


# Demonstrate on a real weight tensor from GPT-2
weight = model_fp32.transformer.h[0].attn.c_attn.weight.data.clone()
print(f"Original weight shape: {weight.shape}")
print(f"Original dtype: {weight.dtype}")
print(f"Value range: [{weight.min():.4f}, {weight.max():.4f}]")
print(f"Original size: {weight.numel() * weight.element_size()} bytes")

# Quantize to different bit widths
for n_bits in [8, 4, 2]:
    q, scale = symmetric_quantize(weight, n_bits)
    dq = symmetric_dequantize(q, scale)
    error = (weight - dq).abs().mean()
    max_error = (weight - dq).abs().max()
    
    print(f"\n{n_bits}-bit symmetric quantization:")
    print(f"  Size: {weight.numel() * n_bits / 8:.0f} bytes ({weight.numel() * weight.element_size() / (weight.numel() * n_bits / 8):.1f}x compression)")
    print(f"  Mean absolute error: {error:.6f}")
    print(f"  Max absolute error:  {max_error:.6f}")
    print(f"  Relative error:      {error / weight.abs().mean():.4%}")

In [ ]:
# Visualize quantization error distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original weight distribution
ax = axes[0, 0]
ax.hist(weight.cpu().numpy().flatten(), bins=100, alpha=0.7, color='steelblue')
ax.set_title('Original Weight Distribution (fp32)')
ax.set_xlabel('Value')
ax.set_ylabel('Count')

# Quantization error for different bit widths
for idx, n_bits in enumerate([8, 4, 2]):
    ax = axes[(idx + 1) // 2, (idx + 1) % 2]
    q, scale = symmetric_quantize(weight, n_bits)
    dq = symmetric_dequantize(q, scale)
    error = (weight - dq).cpu().numpy().flatten()
    
    ax.hist(error, bins=100, alpha=0.7, color=['green', 'orange', 'red'][idx])
    ax.set_title(f'{n_bits}-bit Quantization Error (mean={np.abs(error).mean():.5f})')
    ax.set_xlabel('Error')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Per-channel quantization vs per-tensor (what GPTQ uses)
def per_channel_quantize(weight, n_bits=4, axis=0):
    """
    Quantize each channel (row or column) independently.
    This preserves more precision than per-tensor quantization
    because each channel gets its own scale factor.
    """
    qmin = -(2 ** (n_bits - 1))
    qmax = 2 ** (n_bits - 1) - 1
    
    # Compute per-channel scale
    abs_max = weight.abs().amax(dim=1-axis, keepdim=True)
    scale = abs_max / qmax
    scale = torch.clamp(scale, min=1e-10)  # avoid division by zero
    
    quantized = torch.clamp(torch.round(weight / scale), qmin, qmax).to(torch.int8)
    return quantized, scale


# Compare per-tensor vs per-channel
q_tensor, s_tensor = symmetric_quantize(weight, 4)
dq_tensor = symmetric_dequantize(q_tensor, s_tensor)
error_tensor = (weight - dq_tensor).abs().mean().item()

q_channel, s_channel = per_channel_quantize(weight, 4, axis=0)
dq_channel = q_channel.float() * s_channel
error_channel = (weight - dq_channel).abs().mean().item()

print(f"4-bit per-tensor  error: {error_tensor:.6f}")
print(f"4-bit per-channel error: {error_channel:.6f}")
print(f"Improvement: {(error_tensor - error_channel) / error_tensor:.1%} lower error with per-channel")

In [ ]:
# GPTQ with auto-gptq (if available)
try:
    from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig
    
    print("auto-gptq is available. Example usage:")
    print("""
    # Configure quantization
    quantize_config = BaseQuantizeConfig(
        bits=4,                  # 4-bit quantization
        group_size=128,          # quantize in groups of 128 weights
        desc_act=True,           # order columns by activation magnitude
    )
    
    # Load model and quantize
    model = AutoGPTQForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf", quantize_config)
    model.quantize(calibration_dataset)  # uses ~128 samples
    model.save_quantized("Llama-2-7b-GPTQ")
    
    # Load pre-quantized model
    model = AutoGPTQForCausalLM.from_quantized("TheBloke/Llama-2-7B-GPTQ")
    """)

except ImportError:
    print("auto-gptq not installed. Install with: pip install auto-gptq")
    print("Requires CUDA GPU. We'll continue with conceptual examples.")

---
## 2. AWQ — Activation-Aware Quantization

**AWQ** (Lin et al., 2023) takes a different approach from GPTQ. Instead of optimizing the quantization error directly, AWQ asks: "Which weights matter most?" The answer comes from the activations, not the weights themselves.

The key insight is that a small fraction of weights (about 1%) are disproportionately important because they multiply with large activation values. Quantizing these "salient" weights naively causes significant quality loss, while the other 99% can be aggressively quantized with minimal impact.

AWQ's approach:
1. Run calibration data through the model and record activation magnitudes
2. Identify salient weights: those that correspond to consistently large activations
3. Protect salient weights by multiplying them by a scale factor before quantization, then dividing the activations by the same factor. This effectively gives salient weights more precision.
4. Quantize all weights uniformly (4-bit)

The beauty of AWQ is that the scale factors are determined analytically (not through expensive optimization), making it much faster than GPTQ. It's also more robust — AWQ quantized models often outperform GPTQ at the same bit width, especially at lower precision (3-bit, 2-bit).

In [ ]:
def simulate_awq_vs_naive(weight, activations, n_bits=4):
    """
    Demonstrate the AWQ concept: protect salient weights during quantization.
    
    Args:
        weight: (out_features, in_features) weight matrix
        activations: (n_samples, in_features) activation values from calibration
    """
    # Step 1: Identify salient channels based on activation magnitude
    activation_magnitude = activations.abs().mean(dim=0)  # (in_features,)
    
    # Step 2: Compute per-channel scale factors
    # Channels with large activations get scaled up before quantization
    scale = (activation_magnitude / activation_magnitude.mean()).clamp(min=0.1, max=10.0)
    
    # Naive quantization: quantize weights directly
    q_naive, s_naive = symmetric_quantize(weight, n_bits)
    dq_naive = symmetric_dequantize(q_naive, s_naive)
    
    # AWQ-style: scale weights before quantization
    scaled_weight = weight * scale.unsqueeze(0)  # scale each input channel
    q_awq, s_awq = symmetric_quantize(scaled_weight, n_bits)
    dq_awq = symmetric_dequantize(q_awq, s_awq)
    # Undo scaling after dequantization
    dq_awq = dq_awq / scale.unsqueeze(0)
    
    # Compute output error weighted by activations
    # This measures actual impact on model outputs
    true_output = activations @ weight.T  # (n_samples, out_features)
    naive_output = activations @ dq_naive.T
    awq_output = activations @ dq_awq.T
    
    naive_error = (true_output - naive_output).abs().mean().item()
    awq_error = (true_output - awq_output).abs().mean().item()
    
    return naive_error, awq_error, activation_magnitude


# Use real weights from GPT-2 and simulated activations
weight = model_fp32.transformer.h[0].attn.c_attn.weight.data.clone()

# Simulate realistic activations: most channels have small values,
# but a few channels have consistently large activations (the "outliers")
torch.manual_seed(42)
n_samples = 128
activations = torch.randn(n_samples, weight.shape[0]) * 0.1
# Make ~5% of channels have 10x larger activations
outlier_channels = torch.randperm(weight.shape[0])[:weight.shape[0] // 20]
activations[:, outlier_channels] *= 10.0

naive_err, awq_err, act_mag = simulate_awq_vs_naive(weight, activations, n_bits=4)

print(f"4-bit Naive quantization output error: {naive_err:.6f}")
print(f"4-bit AWQ-style quantization error:    {awq_err:.6f}")
print(f"Improvement: {(naive_err - awq_err) / naive_err:.1%} lower error with AWQ")

In [ ]:
# Visualize which channels AWQ protects
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Activation magnitude per channel
sorted_mag, sorted_idx = act_mag.sort(descending=True)
ax1.bar(range(min(100, len(sorted_mag))), sorted_mag[:100].cpu().numpy(), color='steelblue', alpha=0.7)
ax1.set_xlabel('Channel (sorted by magnitude)')
ax1.set_ylabel('Activation magnitude')
ax1.set_title('Activation Magnitude per Channel (top 100)')
ax1.axhline(y=act_mag.mean().item(), color='red', linestyle='--', label=f'Mean: {act_mag.mean():.3f}')
ax1.legend()

# Compare error across bit widths
bits_range = [2, 3, 4, 6, 8]
naive_errors = []
awq_errors = []
for bits in bits_range:
    ne, ae, _ = simulate_awq_vs_naive(weight, activations, n_bits=bits)
    naive_errors.append(ne)
    awq_errors.append(ae)

x = np.arange(len(bits_range))
width = 0.35
ax2.bar(x - width/2, naive_errors, width, label='Naive', color='red', alpha=0.7)
ax2.bar(x + width/2, awq_errors, width, label='AWQ-style', color='green', alpha=0.7)
ax2.set_xlabel('Quantization bits')
ax2.set_ylabel('Output error')
ax2.set_title('Output Error: Naive vs AWQ-style Quantization')
ax2.set_xticks(x)
ax2.set_xticklabels([str(b) for b in bits_range])
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# AWQ with autoawq (if available)
try:
    from awq import AutoAWQForCausalLM
    print("autoawq is available. Example usage:")
    print("""
    model = AutoAWQForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf")
    tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")
    
    quant_config = {
        "zero_point": True,
        "q_group_size": 128,
        "w_bit": 4,
    }
    model.quantize(tokenizer, quant_config=quant_config)
    model.save_quantized("Llama-2-7b-AWQ")
    """)
except ImportError:
    print("autoawq not installed. Install with: pip install autoawq")
    print("Requires CUDA GPU.")

---
## 3. GGUF — llama.cpp Format

**GGUF** (GPT-Generated Unified Format) is the file format used by llama.cpp, the most popular CPU inference engine for LLMs. Unlike GPTQ and AWQ which target GPU inference, GGUF was designed for efficient CPU execution and can also run on Apple Silicon, AMD GPUs, and even mobile devices.

GGUF's key innovation is **mixed quantization**: different layers can use different precisions. The "K-quant" schemes (introduced by the llama.cpp community) recognize that not all layers are equally sensitive to quantization:

- **Attention layers** tend to be more sensitive, so they get higher precision (e.g., 6-bit)
- **Feed-forward layers** are more robust, so they can use lower precision (e.g., 4-bit)
- **The first and last few layers** are often kept at higher precision

Common K-quant variants:
- `Q4_K_M`: 4-bit with medium quality — good balance of size and quality
- `Q5_K_M`: 5-bit medium — higher quality, slightly larger
- `Q3_K_S`: 3-bit small — maximum compression, noticeable quality loss
- `Q6_K`: 6-bit — near-lossless for most tasks
- `Q2_K`: 2-bit — extreme compression, significant quality loss

The naming convention: `Q{bits}_K_{size}` where size is S(mall), M(edium), or L(arge), indicating how aggressively the less-important layers are quantized.

In [ ]:
# Simulate mixed quantization (the core idea behind K-quants)
def simulate_mixed_quantization(model, layer_bits_map):
    """
    Simulate mixed quantization: different bit widths per layer.
    Returns total size and average error.
    """
    total_params = 0
    total_size_bits = 0
    total_error = 0
    layer_info = []
    
    for name, param in model.named_parameters():
        n_params = param.numel()
        total_params += n_params
        
        # Determine bit width for this layer
        bits = 4  # default
        for pattern, b in layer_bits_map.items():
            if pattern in name:
                bits = b
                break
        
        # Compute quantization error
        q, scale = symmetric_quantize(param.data, bits)
        dq = symmetric_dequantize(q, scale)
        error = (param.data - dq).abs().mean().item()
        
        size_bits = n_params * bits
        total_size_bits += size_bits
        total_error += error * n_params
        
        layer_info.append({
            'name': name,
            'params': n_params,
            'bits': bits,
            'error': error,
        })
    
    return {
        'total_params': total_params,
        'total_size_mb': total_size_bits / 8 / 1e6,
        'avg_bits': total_size_bits / total_params,
        'avg_error': total_error / total_params,
        'layers': layer_info,
    }


# Q4_K_M style: attention at 6-bit, FFN at 4-bit, embeddings at 6-bit
q4_k_m_map = {
    'wte': 6,       # token embeddings
    'wpe': 6,       # position embeddings
    'attn': 6,      # attention layers
    'mlp': 4,       # feed-forward layers
    'ln': 8,        # layer norms (kept at high precision)
}

# Uniform 4-bit
uniform_4bit_map = {}

# Q5_K_M style
q5_k_m_map = {
    'wte': 6,
    'wpe': 6,
    'attn': 6,
    'mlp': 5,
    'ln': 8,
}

configs = {
    'Uniform 4-bit': uniform_4bit_map,
    'Q4_K_M style': q4_k_m_map,
    'Q5_K_M style': q5_k_m_map,
}

fp32_size = sum(p.numel() * 4 for p in model_fp32.parameters()) / 1e6

print(f"Original model (fp32): {fp32_size:.1f} MB\n")
print(f"{'Config':<20} {'Size (MB)':>10} {'Avg bits':>10} {'Avg Error':>12} {'Compression':>12}")
print("=" * 66)

for name, config in configs.items():
    result = simulate_mixed_quantization(model_fp32, config)
    compression = fp32_size / result['total_size_mb']
    print(f"{name:<20} {result['total_size_mb']:>10.1f} {result['avg_bits']:>10.2f} {result['avg_error']:>12.6f} {compression:>11.1f}x")

In [ ]:
# llama-cpp-python usage (if available)
try:
    from llama_cpp import Llama
    print("llama-cpp-python is available. Example usage:")
    print("""
    # Load a GGUF model
    llm = Llama(
        model_path="./models/llama-2-7b.Q4_K_M.gguf",
        n_ctx=2048,
        n_gpu_layers=-1,  # offload all layers to GPU if available
    )
    
    output = llm("The meaning of life is", max_tokens=100, temperature=0.7)
    print(output['choices'][0]['text'])
    """)
except ImportError:
    print("llama-cpp-python not installed. Install with: pip install llama-cpp-python")
    print("Download GGUF models from: https://huggingface.co/TheBloke")

---
## 4. fp8 and int4 — Emerging Precision Formats

The latest GPU hardware introduces native support for reduced-precision formats that can match or exceed the quality of post-training quantization methods while running at near-native speed.

### FP8 (8-bit floating point)

NVIDIA's H100 GPU introduced hardware FP8 support in two variants:
- **E4M3** (4 exponent bits, 3 mantissa bits): More precision, smaller range. Good for weights.
- **E5M2** (5 exponent bits, 2 mantissa bits): Larger range, less precision. Good for gradients.

FP8 maintains the floating-point format (sign + exponent + mantissa), which gives it better representation of the long tails in weight distributions compared to integer quantization. This often translates to better quality at the same bit width.

### INT4 on Blackwell

NVIDIA's Blackwell architecture (B100/B200) introduces native INT4 tensor core operations, making 4-bit inference a first-class citizen. Previously, INT4 required packing (two 4-bit values in one byte) and custom kernels. With native hardware support, INT4 inference approaches the theoretical 2x speedup over INT8.

The practical implication: as hardware catches up, the distinction between "quantized" and "native" inference is blurring. Models that were too large for a single GPU at fp16 can now run at fp8 with minimal quality loss, achieving effectively the same throughput as a model half the size.

In [ ]:
def simulate_fp8_e4m3(tensor):
    """
    Simulate FP8 E4M3 quantization.
    E4M3: 1 sign bit, 4 exponent bits, 3 mantissa bits
    Range: [-448, 448], smallest subnormal: 2^-9
    """
    # E4M3 parameters
    max_val = 448.0
    min_positive = 2**-9  # smallest subnormal
    mantissa_bits = 3
    
    # Clamp to representable range
    clamped = torch.clamp(tensor, -max_val, max_val)
    
    # Simulate reduced precision by rounding mantissa
    # For each value, round to the nearest representable fp8 value
    sign = torch.sign(clamped)
    abs_val = clamped.abs()
    
    # Compute the exponent
    # Add small epsilon to avoid log(0)
    log2_val = torch.log2(abs_val.clamp(min=min_positive))
    exponent = torch.floor(log2_val)
    
    # Round mantissa to 3 bits of precision
    mantissa_scale = 2.0 ** mantissa_bits
    normalized = abs_val / (2.0 ** exponent)
    rounded_mantissa = torch.round(normalized * mantissa_scale) / mantissa_scale
    
    result = sign * rounded_mantissa * (2.0 ** exponent)
    
    # Handle zeros
    result[abs_val < min_positive] = 0.0
    
    return result


def simulate_int8(tensor):
    """Standard INT8 symmetric quantization."""
    q, scale = symmetric_quantize(tensor, 8)
    return symmetric_dequantize(q, scale)


def simulate_int4(tensor):
    """Standard INT4 symmetric quantization."""
    q, scale = symmetric_quantize(tensor, 4)
    return symmetric_dequantize(q, scale)


# Compare formats on GPT-2 weights
weight = model_fp32.transformer.h[5].mlp.c_fc.weight.data.clone()
print(f"Layer: transformer.h[5].mlp.c_fc | Shape: {weight.shape}")
print(f"Value range: [{weight.min():.4f}, {weight.max():.4f}]\n")

formats = {
    'fp32 (original)': weight,
    'fp8 (E4M3)': simulate_fp8_e4m3(weight),
    'int8': simulate_int8(weight),
    'int4': simulate_int4(weight),
}

print(f"{'Format':<20} {'MAE':>12} {'Max Error':>12} {'Rel Error':>12} {'Bits':>6} {'Size (KB)':>10}")
print("=" * 74)
for name, dq in formats.items():
    mae = (weight - dq).abs().mean().item()
    max_err = (weight - dq).abs().max().item()
    rel_err = mae / weight.abs().mean().item()
    bits = {'fp32': 32, 'fp8': 8, 'int8': 8, 'int4': 4}[name.split()[0]]
    size = weight.numel() * bits / 8 / 1024
    print(f"{name:<20} {mae:>12.6f} {max_err:>12.6f} {rel_err:>11.4%} {bits:>6} {size:>10.1f}")

In [ ]:
# Visualize how each format distorts the weight distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (name, dq) in zip(axes.flat, formats.items()):
    vals = dq.cpu().numpy().flatten()
    ax.hist(vals, bins=100, alpha=0.7, color='steelblue', density=True)
    if name != 'fp32 (original)':
        # Overlay original for comparison
        ax.hist(weight.cpu().numpy().flatten(), bins=100, alpha=0.3, color='red',
                density=True, label='Original (fp32)')
        ax.legend()
    ax.set_title(name)
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')

plt.suptitle('Weight Distribution Under Different Precision Formats', fontsize=14)
plt.tight_layout()
plt.show()

---
## 5. Kernel Optimization — How vLLM and TGI Optimize GPU Utilization

Even with quantization, inference speed depends heavily on how efficiently the GPU hardware is utilized. Modern inference engines like vLLM and TGI employ several kernel-level optimizations:

### GPU Occupancy and Memory Bandwidth

LLM inference is typically **memory-bandwidth bound**, not compute-bound. The autoregressive decode step processes just one token at a time, which means the matrix multiplications are thin (one row times the full weight matrix). The GPU's compute units are mostly idle, waiting for data to be fetched from memory. The key metric is **arithmetic intensity** — the ratio of compute operations to memory accesses. For decode, this ratio is very low.

### Fused Kernels

In a standard transformer block, attention involves multiple separate operations: Q/K/V projection, softmax, attention multiply, output projection. Each operation reads from and writes to GPU memory (HBM). **Kernel fusion** combines multiple operations into a single GPU kernel, so intermediate results stay in fast SRAM (registers/shared memory) rather than round-tripping through slow HBM.

FlashAttention is the most famous fused kernel — it computes the entire attention operation in a single pass, reducing memory reads by orders of magnitude.

### vLLM's Approach
- PagedAttention (custom attention kernel that reads from non-contiguous memory pages)
- Continuous batching (dynamic request scheduling)
- Prefix caching (share KV cache for common system prompts)
- Tensor parallelism (split model across GPUs)

### TGI's Approach
- Flash Attention 2
- Custom CUDA kernels for quantized operations
- Token streaming (start returning tokens before generation is complete)
- Speculative decoding (medusa heads)

In [ ]:
# Demonstrate the memory bandwidth bottleneck
def estimate_arithmetic_intensity(batch_size, seq_len, d_model, n_heads, d_head):
    """
    Estimate arithmetic intensity (FLOPS / bytes) for transformer operations.
    Low intensity = memory bound. High intensity = compute bound.
    """
    # For a single attention head:
    # Q*K^T: (batch, seq, d_head) x (batch, d_head, seq) = 2*batch*seq*seq*d_head FLOPS
    # Softmax: ~5*batch*seq*seq FLOPS
    # Attention*V: (batch, seq, seq) x (batch, seq, d_head) = 2*batch*seq*seq*d_head FLOPS
    
    # During decode (seq_len for new token = 1):
    decode_flops = 2 * batch_size * 1 * seq_len * d_head * n_heads  # Q*K^T
    decode_flops += 2 * batch_size * 1 * seq_len * d_head * n_heads  # Attn*V
    
    # Memory reads: need to load full K, V cache
    decode_bytes = 2 * batch_size * seq_len * d_head * n_heads * 2  # K and V, 2 bytes per fp16
    decode_bytes += batch_size * 1 * d_model * 2  # Q vector
    
    decode_intensity = decode_flops / decode_bytes
    
    # During prefill (full seq_len):
    prefill_flops = 2 * batch_size * seq_len * seq_len * d_head * n_heads * 2  # Q*K^T + Attn*V
    prefill_bytes = 2 * batch_size * seq_len * d_head * n_heads * 2 * 3  # Q, K, V
    prefill_intensity = prefill_flops / prefill_bytes
    
    return {
        'decode_intensity': decode_intensity,
        'prefill_intensity': prefill_intensity,
        'decode_flops': decode_flops,
        'prefill_flops': prefill_flops,
    }


# Compare arithmetic intensity for different model sizes
models_config = {
    'GPT-2 (124M)':   {'d_model': 768,  'n_heads': 12, 'd_head': 64},
    'Llama-2 7B':     {'d_model': 4096, 'n_heads': 32, 'd_head': 128},
    'Llama-2 70B':    {'d_model': 8192, 'n_heads': 64, 'd_head': 128},
}

seq_lens = [128, 512, 2048, 8192]

print("Arithmetic Intensity (FLOPS/byte) — higher = more compute-bound")
print("A100 peak: ~312 TFLOPS fp16, ~2 TB/s bandwidth -> optimal intensity ~156\n")

for model_name, config in models_config.items():
    print(f"\n{model_name}:")
    print(f"  {'Seq Len':<10} {'Decode':>12} {'Prefill':>12}")
    for sl in seq_lens:
        result = estimate_arithmetic_intensity(1, sl, **config)
        print(f"  {sl:<10} {result['decode_intensity']:>12.1f} {result['prefill_intensity']:>12.1f}")

In [ ]:
# Visualize: decode is always memory-bound
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for model_name, config in models_config.items():
    decode_intensities = []
    prefill_intensities = []
    for sl in seq_lens:
        result = estimate_arithmetic_intensity(1, sl, **config)
        decode_intensities.append(result['decode_intensity'])
        prefill_intensities.append(result['prefill_intensity'])
    
    ax1.plot(seq_lens, decode_intensities, '-o', label=model_name, linewidth=2)
    ax2.plot(seq_lens, prefill_intensities, '-o', label=model_name, linewidth=2)

# A100 optimal intensity line
optimal = 156  # 312 TFLOPS / 2 TB/s
ax1.axhline(y=optimal, color='red', linestyle='--', label=f'A100 optimal ({optimal})')
ax2.axhline(y=optimal, color='red', linestyle='--', label=f'A100 optimal ({optimal})')

ax1.set_xlabel('Sequence Length')
ax1.set_ylabel('Arithmetic Intensity (FLOPS/byte)')
ax1.set_title('Decode Phase (memory-bound)')
ax1.legend()
ax1.set_xscale('log', base=2)
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Sequence Length')
ax2.set_ylabel('Arithmetic Intensity (FLOPS/byte)')
ax2.set_title('Prefill Phase (compute-bound at long seq)')
ax2.legend()
ax2.set_xscale('log', base=2)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. Benchmarking Inference — Tokens/sec, Time-to-First-Token, Throughput vs Latency

Inference performance has multiple dimensions, and optimizing for one often trades off against another. Understanding these metrics is essential for making deployment decisions.

### Key Metrics

**Time-to-First-Token (TTFT)**: How long from receiving the request until the first generated token is available. This is dominated by the **prefill** phase — the model must process the entire input prompt before generating anything. For long prompts (e.g., RAG with 10K context), TTFT can be seconds. Users perceive TTFT as "how responsive the system feels."

**Tokens per Second (TPS)**: The rate at which tokens are generated after the first token. This is the **decode** phase throughput. For interactive applications, 30+ TPS feels like real-time streaming. Below 10 TPS, users notice the slowness.

**Throughput**: Total tokens generated per second across all concurrent requests. This is the key metric for batch processing and determines cost efficiency. Throughput can be increased by batching more requests together.

**Latency**: Total time to complete a request (TTFT + generation time). There's a fundamental throughput-latency tradeoff: larger batches improve throughput but increase latency for individual requests because each decode step takes longer.

### The Throughput-Latency Tradeoff

This is the most important concept in inference deployment. At batch size 1, you get minimum latency but poor throughput (GPU is underutilized). At large batch sizes, you maximize throughput but individual requests wait longer. The art of inference serving is finding the sweet spot for your use case.

In [ ]:
def benchmark_inference(model, tokenizer, prompt, max_new_tokens=100, n_runs=5):
    """
    Comprehensive inference benchmark measuring TTFT, TPS, and total latency.
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    prompt_len = input_ids.shape[1]
    
    results = []
    
    for run in range(n_runs):
        # Prefill phase — measure TTFT
        start = time.perf_counter()
        with torch.no_grad():
            outputs = model(input_ids, use_cache=True)
            past_kv = outputs.past_key_values
        
        first_token = torch.argmax(outputs.logits[:, -1, :], dim=-1, keepdim=True)
        ttft = time.perf_counter() - start
        
        # Decode phase — measure TPS
        generated = torch.cat([input_ids, first_token], dim=-1)
        decode_start = time.perf_counter()
        n_decoded = 1
        
        for _ in range(max_new_tokens - 1):
            with torch.no_grad():
                outputs = model(
                    generated[:, -1:],
                    past_key_values=past_kv,
                    use_cache=True
                )
                past_kv = outputs.past_key_values
            
            next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=-1)
            n_decoded += 1
            
            if next_token.item() == tokenizer.eos_token_id:
                break
        
        decode_time = time.perf_counter() - decode_start
        total_time = time.perf_counter() - start
        
        results.append({
            'ttft': ttft,
            'decode_time': decode_time,
            'total_time': total_time,
            'tokens_generated': n_decoded,
            'tps': n_decoded / decode_time if decode_time > 0 else 0,
            'inter_token_latency': decode_time / n_decoded * 1000 if n_decoded > 0 else 0,  # ms
        })
    
    return results


# Benchmark with different prompt lengths
prompts = {
    'Short (10 tokens)': "The meaning of life is",
    'Medium (50 tokens)': "In the field of artificial intelligence, researchers have long sought to create systems that can understand and generate human language. The development of large language models has been a significant breakthrough, enabling machines to",
    'Long (repeated)': "The quick brown fox jumps over the lazy dog. " * 20,
}

print(f"{'Prompt':<25} {'TTFT (ms)':>10} {'TPS':>8} {'ITL (ms)':>10} {'Total (ms)':>10}")
print("=" * 65)

all_results = {}
for name, prompt in prompts.items():
    results = benchmark_inference(model_fp32, tokenizer, prompt, max_new_tokens=50, n_runs=3)
    all_results[name] = results
    
    avg = {
        'ttft': np.mean([r['ttft'] for r in results]) * 1000,
        'tps': np.mean([r['tps'] for r in results]),
        'itl': np.mean([r['inter_token_latency'] for r in results]),
        'total': np.mean([r['total_time'] for r in results]) * 1000,
    }
    print(f"{name:<25} {avg['ttft']:>10.1f} {avg['tps']:>8.1f} {avg['itl']:>10.2f} {avg['total']:>10.1f}")

In [ ]:
# Simulate the throughput-latency tradeoff
def simulate_batch_tradeoff(model, tokenizer, prompt, batch_sizes, max_new_tokens=30):
    """
    Measure how batch size affects throughput and latency.
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    results = []
    
    for bs in batch_sizes:
        # Create a batch by repeating the input
        batch_input = input_ids.repeat(bs, 1)
        generated = batch_input.clone()
        
        start = time.perf_counter()
        past_kv = None
        
        for step in range(max_new_tokens):
            with torch.no_grad():
                if past_kv is None:
                    outputs = model(generated, use_cache=True)
                else:
                    outputs = model(generated[:, -1:], past_key_values=past_kv, use_cache=True)
                past_kv = outputs.past_key_values
            
            next_tokens = torch.argmax(outputs.logits[:, -1, :], dim=-1, keepdim=True)
            generated = torch.cat([generated, next_tokens], dim=-1)
        
        elapsed = time.perf_counter() - start
        total_tokens = bs * max_new_tokens
        
        results.append({
            'batch_size': bs,
            'throughput': total_tokens / elapsed,  # tokens/sec across all requests
            'latency': elapsed * 1000,  # ms for the entire batch
            'per_request_latency': elapsed * 1000 / bs,  # approximate
        })
    
    return results


batch_sizes = [1, 2, 4, 8, 16]
if device == "cpu":
    batch_sizes = [1, 2, 4, 8]  # Smaller batches on CPU

prompt = "The future of computing is"
batch_results = simulate_batch_tradeoff(model_fp32, tokenizer, prompt, batch_sizes)

print(f"{'Batch Size':>10} {'Throughput (tok/s)':>18} {'Latency (ms)':>14} {'Per-req (ms)':>14}")
print("=" * 58)
for r in batch_results:
    print(f"{r['batch_size']:>10} {r['throughput']:>18.1f} {r['latency']:>14.1f} {r['per_request_latency']:>14.1f}")

In [ ]:
# Visualize throughput-latency tradeoff
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

bs = [r['batch_size'] for r in batch_results]
throughputs = [r['throughput'] for r in batch_results]
latencies = [r['latency'] for r in batch_results]
per_req = [r['per_request_latency'] for r in batch_results]

# Throughput vs batch size
ax1.plot(bs, throughputs, 'g-o', linewidth=2, markersize=8)
ax1.set_xlabel('Batch Size')
ax1.set_ylabel('Throughput (tokens/sec)')
ax1.set_title('Throughput Scales with Batch Size')
ax1.grid(True, alpha=0.3)

# Latency vs batch size
ax2.plot(bs, latencies, 'r-o', linewidth=2, markersize=8)
ax2.set_xlabel('Batch Size')
ax2.set_ylabel('Total Latency (ms)')
ax2.set_title('Latency Increases with Batch Size')
ax2.grid(True, alpha=0.3)

# Throughput vs latency (the key tradeoff curve)
ax3.plot(latencies, throughputs, 'b-o', linewidth=2, markersize=8)
for i, b in enumerate(bs):
    ax3.annotate(f'bs={b}', (latencies[i], throughputs[i]),
                textcoords="offset points", xytext=(10, 5), fontsize=9)
ax3.set_xlabel('Latency (ms)')
ax3.set_ylabel('Throughput (tokens/sec)')
ax3.set_title('Throughput-Latency Tradeoff')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Comprehensive comparison: fp32 vs simulated quantized model
# We simulate quantization by actually quantizing the weights in-place

def quantize_model_weights(model, n_bits=8):
    """Quantize all weights in a model (in-place simulation)."""
    import copy
    model_q = copy.deepcopy(model)
    
    with torch.no_grad():
        for name, param in model_q.named_parameters():
            if 'weight' in name and param.dim() >= 2:
                q, scale = symmetric_quantize(param.data, n_bits)
                param.data = symmetric_dequantize(q, scale)
    
    return model_q


# Create quantized variants
model_int8 = quantize_model_weights(model_fp32, n_bits=8)
model_int4 = quantize_model_weights(model_fp32, n_bits=4)

# Measure perplexity on a sample text
def compute_perplexity(model, tokenizer, text, stride=512):
    """Compute perplexity using a sliding window approach."""
    encodings = tokenizer(text, return_tensors="pt")
    input_ids = encodings.input_ids.to(device)
    seq_len = input_ids.size(1)
    
    nlls = []
    for begin_loc in range(0, seq_len - 1, stride):
        end_loc = min(begin_loc + 1024, seq_len)
        input_slice = input_ids[:, begin_loc:end_loc]
        target_slice = input_slice.clone()
        
        with torch.no_grad():
            outputs = model(input_slice, labels=target_slice)
            nlls.append(outputs.loss.item())
        
        if end_loc >= seq_len:
            break
    
    return np.exp(np.mean(nlls))


eval_text = """Artificial intelligence has transformed the way we interact with technology. 
From voice assistants to autonomous vehicles, AI systems are becoming increasingly 
integrated into our daily lives. The development of large language models has been 
particularly transformative, enabling new applications in content creation, code 
generation, and scientific research. However, these advances also raise important 
questions about safety, alignment, and the responsible deployment of AI systems."""

print("Computing perplexity for each variant...")
ppl_fp32 = compute_perplexity(model_fp32, tokenizer, eval_text)
ppl_int8 = compute_perplexity(model_int8, tokenizer, eval_text)
ppl_int4 = compute_perplexity(model_int4, tokenizer, eval_text)

print(f"\n{'Variant':<15} {'Perplexity':>12} {'Degradation':>12}")
print("=" * 41)
print(f"{'fp32':<15} {ppl_fp32:>12.2f} {'baseline':>12}")
print(f"{'int8 (sim)':<15} {ppl_int8:>12.2f} {(ppl_int8 - ppl_fp32) / ppl_fp32:>11.2%}")
print(f"{'int4 (sim)':<15} {ppl_int4:>12.2f} {(ppl_int4 - ppl_fp32) / ppl_fp32:>11.2%}")

del model_int8, model_int4  # free memory

---
## Summary

This notebook covered the expert-level topics in inference optimization:

1. **GPTQ**: Post-training quantization using Hessian-guided error compensation. Excellent quality at 4-bit, requires calibration data. Per-channel quantization significantly improves over per-tensor.

2. **AWQ**: Activation-aware quantization that protects salient weights. Often outperforms GPTQ because it focuses on the weights that matter most for model quality, not just minimizing quantization error uniformly.

3. **GGUF**: llama.cpp's mixed quantization format. Different layers get different precision, optimizing the quality-size tradeoff. K-quant schemes (Q4_K_M, Q5_K_M, etc.) provide a range of quality-compression options.

4. **FP8 and INT4**: Hardware-native reduced precision. H100 fp8 and Blackwell int4 blur the line between "quantized" and "native" inference, making extreme compression practical without custom kernels.

5. **Kernel optimization**: Inference is memory-bandwidth bound during decode. Fused kernels (FlashAttention), efficient memory management (PagedAttention), and serving optimizations (continuous batching) are what make vLLM and TGI fast.

6. **Benchmarking**: TTFT, TPS, throughput, and latency measure different aspects of inference performance. The throughput-latency tradeoff is the fundamental deployment decision.

**Next: [build.ipynb](./build.ipynb)** — Hands-on project: quantize a model multiple ways, deploy with vLLM, and comprehensively benchmark.